# Athena · Explore an RSNA holdout case

Compare the reference bone age with the estimates from ResNet-18 and ViT-B/16. The models use saved heads from the benchmark, so this notebook runs on a CPU and needs no training.

Use the two result ZIPs from your benchmark run. The notebook downloads the official RSNA holdout archive, about 1.1 GB, if the runtime doesn't already have it.

For education and research only. Keep notebook outputs containing X-rays private. See the [RSNA challenge and usage terms](https://www.rsna.org/artificial-intelligence/ai-image-challenge/rsna-pediatric-bone-age-challenge-2017).

In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path("/content/Athena-demo")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/n1sh032/Athena.git", str(repo)], check=True)
sys.path.insert(0, str(repo))

holdout = Path("/content/Athena/data/raw/holdout.zip")
if not holdout.exists():
    holdout = Path("/content/rsna-holdout.zip")
if not holdout.exists():
    url = "https://s3.amazonaws.com/east1.public.rsna.org/AI/2017/Bone+Age+Validation+Set.zip"
    partial = holdout.with_suffix(".part")
    subprocess.run(["curl", "-fL", "--retry", "3", "-C", "-", "-o", str(partial), url], check=True)
    partial.replace(holdout)

In [ ]:
from google.colab import files

resnet_zip = Path("/content/rsna-dev-b8-resnet18.zip")
vit_zip = Path("/content/rsna-dev-b8-vit_b_16.zip")
if not (resnet_zip.exists() and vit_zip.exists()):
    print("Select the two result ZIPs from the benchmark.")
    for name, data in files.upload().items():
        target = Path("/content") / Path(name).name
        if target in (resnet_zip, vit_zip):
            target.write_bytes(data)

for path in (resnet_zip, vit_zip):
    if not path.exists():
        raise FileNotFoundError(f"Expected {path.name}. Upload it under that filename.")

In [ ]:
from IPython.display import Image, Markdown, display
from scripts.demo import demo

image_id = None  # Use the middle ID by default, or enter a specific holdout ID.
case_id, reference, png, estimates = demo(holdout, resnet_zip, vit_zip, image_id)

display(Markdown(f"### RSNA holdout image {case_id}\n\nReference bone age: **{reference} months**"))
display(Image(data=png, width=320))
table = ["| Model | Estimate | Absolute error |", "|---|---:|---:|"]
for name, estimate in estimates.items():
    table.append(f"| {name} | {estimate:.1f} months | {abs(estimate - reference):.1f} months |")
display(Markdown("\n".join(table)))